# Notebook 1: Data Understanding and Preprocessing
## CS336 - Artificial Intelligence and Machine Learning (AIML)
### Assignment: Smart Energy Consumption Advisory Agent

**Purpose:** This notebook performs the initial exploration and cleaning of the smart meter dataset.
It covers loading data, unde...

In [ ]:
# ─── Standard library and third-party imports ───────────────────────────────
import numpy as np                  # Numerical computing
import pandas as pd                 # Data manipulation and analysis
import matplotlib.pyplot as plt     # Plotting library
import seaborn as sns               # Statistical visualizations
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore')

# Set consistent plotting aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully.')

## 1. Dataset Loading

We use a **synthetic smart meter dataset** that mirrors the structure of the UCI/Kaggle 
Individual Household Electric Power Consumption dataset.

In [ ]:
# ─── Synthetic data generation (mirrors UCI smart meter schema) ──────────────
np.random.seed(42)  # Fix random seed for reproducibility

n = 35_040  # 15-min intervals → 1 full year of data

# Generate a continuous time index spanning one year
timestamps = pd.date_range(start='2023-01-01', periods=n, freq='15min')

# Hour of day (0-23) drives the diurnal usage pattern
hour = timestamps.hour

# Base active power follows a sinusoidal pattern peaking in evenings
base_power = 1.5 + 1.2 * np.sin((hour - 6) * np.pi / 12)

# Add realistic Gaussian noise to simulate appliance variation
noise = np.random.normal(0, 0.3, n)

# Inject a small fraction of outlier spikes (e.g., faulty readings)
outlier_mask = np.random.rand(n) < 0.005      # 0.5 % of readings
outlier_values = np.random.uniform(8, 12, n)  # Unrealistically high power

global_active_power = np.where(outlier_mask,
                               outlier_values,
                               np.clip(base_power + noise, 0.1, 7.0))

# Sub-metering channels (proportional fractions of total, with noise)
sub1 = np.clip(global_active_power * 0.15 * 1000 / 60 + np.random.normal(0, 2, n), 0, None)
sub2 = np.clip(global_active_power * 0.10 * 1000 / 60 + np.random.normal(0, 1.5, n), 0, None)
sub3 = np.clip(global_active_power * 0.30 * 1000 / 60 + np.random.normal(0, 3, n), 0, None)

# Introduce ~1 % missing values in each numeric column to simulate sensor dropouts
def inject_nans(arr, frac=0.01):
    arr = arr.astype(float).copy()
    idx = np.random.choice(len(arr), int(len(arr) * frac), replace=False)
    arr[idx] = np.nan
    return arr

df = pd.DataFrame({
    'timestamp':            timestamps,
    'global_active_power':  inject_nans(global_active_power),
    'sub_metering_1':       inject_nans(sub1),
    'sub_metering_2':       inject_nans(sub2),
    'sub_metering_3':       inject_nans(sub3),
    'voltage':              inject_nans(np.random.normal(240, 2, n)),
    'global_intensity':     inject_nans(global_active_power * 1000 / 240),
})

df.set_index('timestamp', inplace=True)  # Treat timestamp as the DataFrame index

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Overview

In [ ]:
# ─── Basic structural summary ────────────────────────────────────────────────
# dtypes() reveals whether columns are parsed correctly
print('── Column Data Types ──')
print(df.dtypes)

print('\n── Descriptive Statistics ──')
# describe() shows count, mean, std, min, quartiles, max for each numeric column
df.describe().round(3)

In [ ]:
# ─── Missing-value audit ─────────────────────────────────────────────────────
# Understanding the extent of missingness guides imputation strategy
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print(missing_summary)

# Visualise missingness as a bar chart
missing_pct[missing_pct > 0].plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Missing Value Percentage per Column')
plt.ylabel('% Missing')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Missing-Value Imputation

We choose **time-based forward fill** followed by backward fill for residual gaps.

In [ ]:
# ─── Imputation: forward fill → backward fill ────────────────────────────────
# ffill propagates the last valid reading forward (handles short dropouts well)
df.ffill(inplace=True)

# bfill handles any remaining NaNs at the very start of the series
df.bfill(inplace=True)

# Confirm no missing values remain
assert df.isnull().sum().sum() == 0, 'Imputation incomplete — NaNs still present!'
print('Missing value imputation complete. No NaNs remain.')

## 4. Outlier Detection and Removal

We apply the **IQR fence method** (1.5× IQR) to the main power column.

In [ ]:
# ─── IQR-based outlier removal on global_active_power ────────────────────────
col = 'global_active_power'

Q1 = df[col].quantile(0.25)   # First quartile
Q3 = df[col].quantile(0.75)   # Third quartile
IQR = Q3 - Q1                 # Inter-quartile range

lower_fence = Q1 - 1.5 * IQR  # Values below this are extreme low outliers
upper_fence = Q3 + 1.5 * IQR  # Values above this are extreme high outliers

print(f'Q1={Q1:.3f}, Q3={Q3:.3f}, IQR={IQR:.3f}')
print(f'Fence: [{lower_fence:.3f}, {upper_fence:.3f}]')

# Retain only readings within the IQR fences
before = len(df)
df = df[(df[col] >= lower_fence) & (df[col] <= upper_fence)]
after = len(df)

print(f'Rows before: {before} | Rows after: {after} | Removed: {before - after} ({(before-after)/before*100:.2f}%)')

## 5. Feature Engineering — Temporal Attributes

Temporal features allow models to capture **time-of-day**, **day-of-week**, and **seasonal** effects.

In [ ]:
# ─── Derive temporal features from the timestamp index ───────────────────────
df['hour']        = df.index.hour           # 0–23: captures diurnal pattern
df['day_of_week'] = df.index.dayofweek      # 0=Monday … 6=Sunday
df['month']       = df.index.month          # 1–12: captures seasonal variation
df['is_weekend']  = (df.index.dayofweek >= 5).astype(int)  # Binary weekend flag

# Cyclical encoding: hour and month are circular (23→0, Dec→Jan)
# Sine/cosine encoding preserves this circular continuity for ML models
df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print('New feature columns added:')
print([c for c in df.columns if c not in ['global_active_power', 'sub_metering_1',
       'sub_metering_2', 'sub_metering_3', 'voltage', 'global_intensity']])
df.head(3)

## 6. Normalisation

**Min-Max scaling** maps each feature to [0, 1].

In [ ]:
# ─── Min-Max normalisation of numeric columns ────────────────────────────────
from sklearn.preprocessing import MinMaxScaler  # scikit-learn scaler

# Select only the raw sensor columns for normalisation
scale_cols = ['global_active_power', 'sub_metering_1',
              'sub_metering_2', 'sub_metering_3', 'voltage', 'global_intensity']

scaler = MinMaxScaler()               # Initialise the scaler object
df_scaled = df.copy()                 # Work on a copy to preserve originals
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])  # Fit and transform

print('Scaled statistics (should be in [0, 1]):')
df_scaled[scale_cols].describe().loc[['min', 'max']].round(4)

## 7. Distribution Visualisation

In [ ]:
# ─── Histograms: raw vs. scaled global_active_power ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution — shows natural skew and spread
axes[0].hist(df['global_active_power'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Raw Global Active Power (kW)', fontsize=13)
axes[0].set_xlabel('Power (kW)')
axes[0].set_ylabel('Frequency')

# Scaled distribution — confirms values are in [0, 1]
axes[1].hist(df_scaled['global_active_power'], bins=60, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Normalised Global Active Power', fontsize=13)
axes[1].set_xlabel('Normalised Power [0–1]')
axes[1].set_ylabel('Frequency')

plt.suptitle('Power Consumption Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
# ─── Pearson correlation matrix ───────────────────────────────────────────────
# Strong correlations among sub_metering and global_active_power are expected
# because sub-meters measure portions of the same circuit
corr = df[scale_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Pearson Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Save Clean Dataset

In [ ]:
# ─── Persist preprocessed data for use in subsequent notebooks ───────────────
# We save both the original-scale cleaned data and the normalised version.
df.to_csv('data_clean.csv')           # Used for human-readable analysis
df_scaled.to_csv('data_scaled.csv')   # Used by clustering and anomaly models

print('Saved: data_clean.csv')
print('Saved: data_scaled.csv')
print(f'Final dataset: {df.shape[0]} rows × {df.shape[1]} columns')